In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.impute import SimpleImputer

# Configuration des visualisations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Ignorer les warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Chargement du fichier CSV
df = pd.read_csv('heart_disease_uci.csv')

print("=" * 60)
print("APERÇU DES DONNÉES")
print("=" * 60)
print(f"Shape du dataset: {df.shape}")
print(f"\nNombre de colonnes: {df.shape[1]}")
print(f"Nombre de lignes: {df.shape[0]}")

print("\n" + "=" * 60)
print("PREMIÈRES LIGNES")
print("=" * 60)
df.head(10)

In [ ]:
print("=" * 60)
print("INFORMATIONS SUR LES COLONNES")
print("=" * 60)
df.info()

In [ ]:
print("=" * 60)
print("STATISTIQUES DESCRIPTIVES")
print("=" * 60)
df.describe()

In [ ]:
# Vérification des valeurs manquantes
print("=" * 60)
print("VALEURS MANQUANTES PAR COLONNE")
print("=" * 60)
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({
    'Valeurs manquantes': missing_values,
    'Pourcentage (%)': missing_percent
})
missing_df = missing_df[missing_df['Valeurs manquantes'] > 0].sort_values('Valeurs manquantes', ascending=False)
print(missing_df)

print(f"\nTotal des valeurs manquantes: {df.isnull().sum().sum()}")

In [ ]:
# Analyse de la variable cible 'num'
print("=" * 60)
print("ANALYSE DE LA VARIABLE CIBLE (num)")
print("=" * 60)

# Distribution des valeurs de num
print("\nDistribution des classes:")
print(df['num'].value_counts().sort_index())

# Statistiques
print(f"\nPourcentage de patients avec maladie cardiaque (num > 0): {((df['num'] > 0).sum() / len(df)) * 100:.2f}%")
print(f"Pourcentage de patients sans maladie cardiaque (num = 0): {((df['num'] == 0).sum() / len(df)) * 100:.2f}%")

# Visualisation de la distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution des classes
ax1 = axes[0]
df['num'].value_counts().sort_index().plot(kind='bar', ax=ax1, color='steelblue', edgecolor='black')
ax1.set_xlabel('Num (0 = absence, 1-4 = présence)', fontsize=12)
ax1.set_ylabel('Nombre de patients', fontsize=12)
ax1.set_title('Distribution de la variable cible', fontsize=14, fontweight='bold')
ax1.set_xticklabels(['0 (Absence)', '1', '2', '3', '4'], rotation=0)

# Binaire pour la classification
ax2 = axes[1]
df['target_binary'] = (df['num'] > 0).astype(int)
df['target_binary'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['lightgreen', 'salmon'], explode=[0.05, 0], ax=ax2)
ax2.set_ylabel('')
ax2.set_title('Distribution binaire (0 = Sain, 1 = Malade)', fontsize=14, fontweight='bold')
ax2.legend(['Sain', 'Malade'], loc='upper right')

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)

In [ ]:
# Variables numériques à analyser
numeric_features = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, feature in enumerate(numeric_features):
    ax = axes[idx]
    # Distribution par statut
    for status in [0, 1]:
        data = df[df['target_binary'] == status][feature].dropna()
        ax.hist(data, alpha=0.6, label=f"{'Malade' if status == 1 else 'Sain'}", bins=20)
    ax.set_xlabel(feature, fontsize=11)
    ax.set_ylabel('Fréquence', fontsize=11)
    ax.set_title(f'Distribution de {feature}', fontsize=12, fontweight='bold')
    ax.legend()
    
# Boîtes à moustaches
ax = axes[5]
df_melted = df.melt(id_vars=['target_binary'], value_vars=numeric_features, var_name='feature', value_name='value')
df_melted = df_melted.dropna()
sns.boxplot(data=df_melted, x='feature', y='value', hue='target_binary', ax=ax)
ax.set_title('Distribution des variables numériques', fontsize=12, fontweight='bold')
ax.set_xticklabels(numeric_features, rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Variables catégorielles
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, feature in enumerate(categorical_features):
    if idx < len(categorical_features):
        ax = axes[idx]
        # Créer un tableau croisé
        cross_tab = pd.crosstab(df[feature], df['target_binary'], normalize='index') * 100
        cross_tab.plot(kind='bar', stacked=True, ax=ax, color=['lightgreen', 'salmon'])
        ax.set_xlabel(feature, fontsize=11)
        ax.set_ylabel('Pourcentage (%)', fontsize=11)
        ax.set_title(f'Maladie cardiaque par {feature}', fontsize=12, fontweight='bold')
        ax.legend(['Sain', 'Malade'], loc='upper right')
        ax.tick_params(axis='x', rotation=45)
        
# Supprimer le dernier subplot vide
if len(categorical_features) < len(axes):
    fig.delaxes(axes[-1])

plt.tight_layout()
plt.show()

In [ ]:
# Sélection des colonnes numériques pour la corrélation
numeric_df = df.select_dtypes(include=[np.number]).copy()
numeric_df['target_binary'] = (df['num'] > 0).astype(int)

# Matrice de corrélation
plt.figure(figsize=(14, 12))
correlation_matrix = numeric_df.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r', 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Matrice de corrélation des variables numériques', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Corrélation avec la cible
print("=" * 60)
print("CORRÉLATION AVEC LA VARIABLE CIBLE")
print("=" * 60)
target_corr = correlation_matrix['target_binary'].sort_values(ascending=False)
for var, corr in target_corr.items():
    if var != 'target_binary':
        print(f"{var:15} : {corr:.3f}")

In [ ]:
# Création d'une copie pour le prétraitement
df_processed = df.copy()
df_processed['target'] = (df_processed['num'] > 0).astype(int)

print("=" * 60)
print("PRÉTRAITEMENT DES DONNÉES")
print("=" * 60)

# 1. Gestion des colonnes à supprimer
columns_to_drop = ['id', 'num', 'target_binary']
df_processed = df_processed.drop(columns=[col for col in columns_to_drop if col in df_processed.columns])

print(f"Colonnes après suppression: {df_processed.shape[1]}")
print(f"Variables disponibles: {list(df_processed.columns)}")

In [ ]:
# Identification des colonnes avec valeurs manquantes
print("\n" + "-" * 40)
print("GESTION DES VALEURS MANQUANTES")
print("-" * 40)

missing_before = df_processed.isnull().sum()
print(f"Valeurs manquantes avant imputation:\n{missing_before[missing_before > 0]}")

# Séparation des features numériques et catégorielles
numeric_cols = df_processed.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()

print(f"\nColonnes numériques: {numeric_cols}")
print(f"Colonnes catégorielles: {categorical_cols}")

# Imputation pour colonnes numériques (par la médiane)
if 'ca' in numeric_cols and df_processed['ca'].isnull().sum() > 0:
    df_processed['ca'] = df_processed['ca'].fillna(df_processed['ca'].median())

# Imputation pour colonnes catégorielles (par le mode)
for col in categorical_cols:
    if df_processed[col].isnull().sum() > 0:
        mode_value = df_processed[col].mode()[0]
        df_processed[col] = df_processed[col].fillna(mode_value)

# Vérification des valeurs manquantes restantes
missing_after = df_processed.isnull().sum()
print(f"\nValeurs manquantes après imputation:\n{missing_after[missing_after > 0]}")
if missing_after.sum() == 0:
    print("✓ Aucune valeur manquante restante!")

In [ ]:
print("\n" + "-" * 40)
print("ENCODAGE DES VARIABLES CATÉGORIELLES")
print("-" * 40)

# Encodage des variables binaires et catégorielles
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col].astype(str))
    label_encoders[col] = le
    print(f"{col:15} -> {len(le.classes_)} catégories: {le.classes_}")

print(f"\nShape après encodage: {df_processed.shape}")

In [ ]:
print("\n" + "-" * 40)
print("SÉPARATION FEATURES / CIBLE")
print("-" * 40)

X = df_processed.drop('target', axis=1)
y = df_processed['target']

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print(f"\nDistribution de y:")
print(y.value_counts())
print(f"Classe 0 (sain): {(y == 0).sum()}")
print(f"Classe 1 (malade): {(y == 1).sum()}")

In [ ]:
# Division des données
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain set: {X_train.shape[0]} échantillons")
print(f"Test set: {X_test.shape[0]} échantillons")

# Standardisation (mise à l'échelle)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✓ Standardisation effectuée avec succès")

In [ ]:
print("=" * 60)
print("ENTRAÎNEMENT DU MODÈLE")
print("=" * 60)

# Création et entraînement du modèle
model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)

print("Modèle de régression logistique entraîné avec succès")
print(f"Coefficients du modèle: {len(model.coef_[0])} features")

In [ ]:
# Prédictions
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

# Résumé des prédictions
print("\n" + "-" * 40)
print("RÉSULTATS DES PRÉDICTIONS")
print("-" * 40)
print(f"Nombre total de prédictions: {len(y_pred)}")
print(f"Prédictions classe 0 (sain): {(y_pred == 0).sum()}")
print(f"Prédictions classe 1 (malade): {(y_pred == 1).sum()}")

In [ ]:
print("=" * 60)
print("ÉVALUATION DU MODÈLE")
print("=" * 60)

# Métriques
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\nMÉTRIQUES PRINCIPALES:")
print("-" * 30)
print(f"Exactitude (Accuracy):  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Précision (Precision):  {precision:.4f} ({precision*100:.2f}%)")
print(f"Rappel (Recall):        {recall:.4f} ({recall*100:.2f}%)")
print(f"Score F1:               {f1:.4f}")

print("\n" + "-" * 40)
print("RAPPORT DE CLASSIFICATION")
print("-" * 40)
print(classification_report(y_test, y_pred, target_names=['Sain (0)', 'Malade (1)']))

In [ ]:
# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matrice de confusion simple
ax1 = axes[0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1, 
            xticklabels=['Sain (0)', 'Malade (1)'],
            yticklabels=['Sain (0)', 'Malade (1)'])
ax1.set_xlabel('Prédictions', fontsize=12)
ax1.set_ylabel('Vérité terrain', fontsize=12)
ax1.set_title('Matrice de confusion', fontsize=14, fontweight='bold')

# Matrice de confusion normalisée
ax2 = axes[1]
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='YlOrRd', ax=ax2,
            xticklabels=['Sain (0)', 'Malade (1)'],
            yticklabels=['Sain (0)', 'Malade (1)'])
ax2.set_xlabel('Prédictions', fontsize=12)
ax2.set_ylabel('Vérité terrain', fontsize=12)
ax2.set_title('Matrice de confusion normalisée', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nINTERPRÉTATION DE LA MATRICE DE CONFUSION:")
print(f"Vrais négatifs (TN):  {cm[0,0]} - Patients sains correctement identifiés")
print(f"Faux positifs (FP):   {cm[0,1]} - Patients sains prédits malades")
print(f"Faux négatifs (FN):   {cm[1,0]} - Patients malades prédits sains")
print(f"Vrais positifs (TP):   {cm[1,1]} - Patients malades correctement identifiés")

In [ ]:
# Analyse des features importantes
feature_names = X.columns
coefficients = model.coef_[0]

# Création du DataFrame des coefficients
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Abs_Coefficient': np.abs(coefficients)
}).sort_values('Abs_Coefficient', ascending=False)

print("=" * 60)
print("ANALYSE DES FEATURES IMPORTANTES")
print("=" * 60)
print("\nTop 10 des features les plus influentes:")
print(coef_df.head(10).to_string(index=False))

# Visualisation des coefficients
plt.figure(figsize=(12, 8))
top_features = coef_df.head(15)
colors = ['red' if x < 0 else 'green' for x in top_features['Coefficient']]
plt.barh(top_features['Feature'], top_features['Coefficient'], color=colors, edgecolor='black')
plt.xlabel('Coefficient (impact sur la prédiction)', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Importance des features - Régression Logistique\n(vert = corrélation positive, rouge = corrélation négative)', 
          fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Visualisation des métriques
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique des métriques
ax1 = axes[0]
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
values = [accuracy, precision, recall, f1]
colors_bar = ['#2ecc71', '#3498db', '#e74c3c', '#9b59b6']
bars = ax1.bar(metrics, values, color=colors_bar, edgecolor='black', linewidth=1.5)
ax1.set_ylim(0, 1)
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Métriques d\'évaluation du modèle', fontsize=14, fontweight='bold')
for bar, val in zip(bars, values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
ax1.axhline(y=0.7, color='red', linestyle='--', alpha=0.5, label='Seuil 0.7')
ax1.legend()

# Courbe ROC (simulée à partir des probabilités)
from sklearn.metrics import roc_curve, auc
ax2 = axes[1]
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)
ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
ax2.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Aléatoire (AUC = 0.5)')
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('Taux de faux positifs', fontsize=12)
ax2.set_ylabel('Taux de vrais positifs', fontsize=12)
ax2.set_title('Courbe ROC', fontsize=14, fontweight='bold')
ax2.legend(loc="lower right")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAUC (Area Under Curve): {roc_auc:.4f}")

In [ ]:
print("=" * 60)
print("RÉSUMÉ ET CONCLUSIONS")
print("=" * 60)

print("""
 RÉSULTATS PRINCIPAUX:
------------------------
1. Le modèle de régression logistique atteint une exactitude de {:.2f}%
2. Le rappel (recall) de {:.2f}% indique que le modèle identifie bien les patients malades
3. La précision de {:.2f}% montre que les prédictions de maladie sont fiables
4. Le score F1 de {:.3f} représente un bon équilibre entre précision et rappel

 FEATURES LES PLUS IMPORTANTES:
------------------------
""".format(accuracy*100, recall*100, precision*100, f1))

for i, row in coef_df.head(5).iterrows():
    effect = "augmente" if row['Coefficient'] > 0 else "diminue"
    print(f"   • {row['Feature']:12} : {effect} le risque (coeff = {row['Coefficient']:.3f})")

print("""
 INTERPRÉTATION MÉDICALE:
------------------------
- Les variables avec coefficient positif sont des facteurs de risque
- Les variables avec coefficient négatif sont des facteurs protecteurs
- L'âge, les douleurs thoraciques atypiques et l'ECG anormal sont parmi les prédicteurs majeurs

 LIMITES ET AMÉLIORATIONS POSSIBLES:
------------------------
1. Données déséquilibrées - utilisation de class_weight='balanced' pour compenser
2. Variables manquantes traitées par imputation (médiane/mode)
3. Possibilité d'expérimenter d'autres algorithmes (Random Forest, XGBoost)
4. Validation croisée pour une meilleure généralisation
""")